# SHAP Explanations → CSV

This tutorial walks through generating SHAP explanations for the CoAX `wine_quality`
dataset and saving them to `tutorials/generated_explanation/` in the standard
explanation CSV format used throughout this project.

**Covered methods**

| Registry key | Class | Algorithm | Model type |
|---|---|---|---|
| `shap` / `shap_kernel` | `KernelShap` | Weighted linear regression with Shapley kernel | Any black-box |
| `shap_tree` | `ShapTreeExplainer` | Exact polynomial-time tree traversal | RF / XGBoost / LightGBM |
| `shap_linear` | `ShapLinearExplainer` | Closed-form via feature covariance | Linear / logistic regression |

KernelSHAP is the project default — `create_xai_method("shap", ...)` resolves to it.
TreeSHAP and LinearSHAP are faster exact alternatives when the model type is known.

**Output CSV schema** (matches `assets/explanations/coax/attribution.csv`)
```
dataId, modelName, expMethod, instanceId, pred, i_max, a0_i … aN_i, intercept
```
- `aN_i`     — attribution for feature N, normalized by `i_max`
- `i_max`    — max absolute raw attribution for this instance (normalization scale)
- `intercept`— SHAP base value (expected model output)

## 1 · Setup

In [9]:
from pathlib import Path
import sys

repo_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "src").exists() and (candidate / "assets").exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.xai_adapter import KernelShap, ShapLinearExplainer, ShapTreeExplainer, create_xai_method

OUTPUT_DIR = repo_root / "tutorials" / "generated_explanation"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"repo_root:    {repo_root}")
print(f"output_dir:   {OUTPUT_DIR}")

repo_root:    /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api
output_dir:   /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/tutorials/generated_explanation


## 2 · Load CoAX Data

The CoAX assets live in `assets/ai_dataset/coax/`.  We load the three files directly
with pandas and select the `wine_quality` dataset.

- `values.csv`   — raw feature values (`x0`–`x4`) and ground-truth label `y`
- `metadata.csv` — human-readable feature names and min/max bounds per dataset
- `none.csv`     — model predictions (`pred`) keyed by `dataId, modelName, instanceId`

In [2]:
ASSETS     = repo_root / "assets" / "ai_dataset" / "coax"
DATASET_ID = "wine_quality"

values_df   = pd.read_csv(ASSETS / "values.csv")
metadata_df = pd.read_csv(ASSETS / "metadata.csv")
none_df     = pd.read_csv(ASSETS / "none.csv")

data = values_df[values_df["dataId"] == DATASET_ID].reset_index(drop=True)
meta = metadata_df[metadata_df["dataId"] == DATASET_ID].iloc[0]

# Feature columns are x0, x1, … (exclude the label 'y')
feature_cols  = [c for c in data.columns if c.startswith("x") and c != "y"]
feature_names = [str(meta[col]) for col in feature_cols]

X            = data[feature_cols].values.astype(float)
y            = data["y"].values.astype(int)
instance_ids = data["instanceId"].values

print(f"Dataset : {DATASET_ID}")
print(f"Shape   : {X.shape}")
print(f"Features: {list(zip(feature_cols, feature_names))}")
print(f"Classes : {dict(zip(*np.unique(y, return_counts=True)))}")
data.head()

Dataset : wine_quality
Shape   : (120, 5)
Features: [('x0', 'Vinegar Taint'), ('x1', 'SO2'), ('x2', 'pH'), ('x3', 'Sulphates'), ('x4', 'Alcohol')]
Classes : {np.int64(0): np.int64(64), np.int64(1): np.int64(56)}


,dataId,instanceId,x0,x1,x2,x3,x4,y
0,wine_quality,0,0.50,17.0,3.32,0.71,10.5,0
1,wine_quality,1,0.58,55.0,3.46,0.59,10.2,0
2,wine_quality,2,0.46,98.0,3.33,0.62,9.8,0
3,wine_quality,3,0.60,10.0,3.18,0.63,10.4,0
4,wine_quality,4,0.33,80.0,3.30,0.76,10.7,1


## 3 · Train Models

We train two models on the same data:

- **Random Forest** — tree-based, scale-invariant. Used for KernelSHAP and TreeSHAP.
- **Logistic Regression** — linear. Requires standardized features. Used for LinearSHAP.

KernelSHAP and TreeSHAP are run on the **same RF model** so their values can be
directly compared — TreeSHAP is the exact answer; KernelSHAP is an approximation.

In [3]:
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, instance_ids, test_size=0.2, random_state=42
)

# StandardScaler for LR (RF is scale-invariant)
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f"RF  accuracy: {rf.score(X_test, y_test):.3f}")

lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_sc, y_train)
print(f"LR  accuracy: {lr.score(X_test_sc, y_test):.3f}")

RF  accuracy: 0.875
LR  accuracy: 0.750


## 4 · Explanation → CSV

Every SHAP adapter returns an `XAIAdapterResult` with:
- `result.values`      — `(n_instances, n_features)` attribution array
- `result.base_values` — `(n_instances,)` SHAP base values (expected model output)
- `result.method`      — adapter method name (e.g. `"shap_kernel"`)

`XAIAdapterResult` ships two built-in conversion methods:

| Method | Returns | Description |
|---|---|---|
| `result.to_explanation_df(instance_ids, predictions, dataset_id, model_name)` | `pd.DataFrame` | Standard explanation schema |
| `result.save_csv(path, instance_ids, predictions, dataset_id, model_name)` | `None` | Saves to CSV, creates parent dirs |

Schema: `dataId, modelName, expMethod, instanceId, pred, i_max, a0_i … aN_i, intercept`
- `aN_i` — attribution normalized by `i_max` (max absolute per instance)
- `intercept` — SHAP base value

## 5 · KernelSHAP (default `"shap"`)

`create_xai_method("shap", ...)` resolves to `KernelShap` — the model-agnostic
default. It only needs a `predict_fn`, so it works with any black-box model.

Internally it uses `shap.KernelExplainer`, which approximates Shapley values by
fitting a weighted linear model on perturbed feature coalitions. It is slower than
TreeSHAP but is not limited to a specific model family.

`.fit(background_data)` is called automatically when `background_data` is passed
to the constructor.

In [7]:
ks_df = None; ks_result = None; ks_preds = None
try:
    kernel_shap = create_xai_method(
        "shap",                       # alias for shap_kernel / KernelShap
        predict_fn=rf.predict_proba,
        background_data=X_train,
        n_background_samples=80,
        target=1,
    )

    ks_result = kernel_shap.explain(X_test)
    ks_preds  = rf.predict(X_test)
    ks_df     = ks_result.to_explanation_df(ids_test, ks_preds, DATASET_ID, "rf")

    print(f"KernelSHAP: {ks_df.shape[0]} instances, {ks_df.shape[1]} columns")
    ks_df.head(3)

except ImportError as exc:
    print(f"Skipped — install shap: pip install shap  ({exc})")

100%|██████████| 24/24 [00:00<00:00, 106.53it/s]

KernelSHAP: 24 instances, 12 columns


## 6 · ShapTreeExplainer

`ShapTreeExplainer` wraps `shap.TreeExplainer`.  It exploits the tree structure to
compute **exact** Shapley values in polynomial time — no sampling needed.

The default `feature_perturbation="auto"` selects `"tree_path_dependent"` when no
background data is given.  This is the correct mode for sklearn RandomForest: it uses
the training-data distribution already encoded in the trees, produces exact values, and
avoids the approximation error that `"interventional"` mode accumulates when
marginalizing over a separate background set.

Pass `background_data` and `feature_perturbation="interventional"` only if you
explicitly need interventional Shapley values (useful for removing spurious correlations
between features, but approximate for RF and can fail the additivity check).

Because it runs on the **same RF model**, you can compare values directly to KernelSHAP:
TreeSHAP is ground truth; KernelSHAP should rank features similarly.

In [6]:
tree_df = None; tree_result = None
try:
    # No background_data: "auto" → "tree_path_dependent" for RF.
    # Uses the training distribution already in the trees — exact, no sampling error.
    tree_shap   = ShapTreeExplainer(model=rf, target=1)
    tree_result = tree_shap.explain(X_test)
    tree_df     = tree_result.to_explanation_df(ids_test, ks_preds, DATASET_ID, "rf")

    print(f"TreeSHAP:   {tree_df.shape[0]} instances, {tree_df.shape[1]} columns")
    tree_df.head(3)

except ImportError as exc:
    print(f"Skipped — install shap: pip install shap  ({exc})")

TreeSHAP:   24 instances, 12 columns


## 7 · ShapLinearExplainer

`ShapLinearExplainer` wraps `shap.LinearExplainer`.  It uses the model's coefficients
and the feature covariance matrix to compute **exact** Shapley values for linear models.

The background data and the instances passed to `.explain()` must be in the **same
normalized space** the model was trained on — here that is StandardScaler output.

In [7]:
linear_df = None; linear_result = None; lr_preds = None
try:
    linear_shap   = ShapLinearExplainer(model=lr, background_data=X_train_sc, nsamples=200, target=1)
    linear_result = linear_shap.explain(X_test_sc)
    lr_preds      = lr.predict(X_test_sc)
    linear_df     = linear_result.to_explanation_df(ids_test, lr_preds, DATASET_ID, "lr")

    print(f"LinearSHAP: {linear_df.shape[0]} instances, {linear_df.shape[1]} columns")
    linear_df.head(3)

except ImportError as exc:
    print(f"Skipped — install shap: pip install shap  ({exc})")

LinearSHAP: 24 instances, 12 columns


## 8 · Compare Mean |Attribution| Across Methods

Average absolute normalized attribution per feature.  KernelSHAP and TreeSHAP should
rank features similarly (same RF model); LinearSHAP (LR) may differ because LR
captures only linear effects.

In [8]:
attr_cols = [f"a{k}_i" for k in range(len(feature_cols))]

rows = {"feature": feature_names}
if ks_df     is not None: rows["KernelSHAP (RF)"]  = ks_df[attr_cols].abs().mean().values
if tree_df   is not None: rows["TreeSHAP (RF)"]    = tree_df[attr_cols].abs().mean().values
if linear_df is not None: rows["LinearSHAP (LR)"] = linear_df[attr_cols].abs().mean().values

pd.DataFrame(rows).sort_values("KernelSHAP (RF)" if ks_df is not None else "TreeSHAP (RF)",
                                ascending=False).reset_index(drop=True)

,feature,KernelSHAP (RF),TreeSHAP (RF),LinearSHAP (LR)
0,Sulphates,0.922010,0.947865,0.848085
1,Alcohol,0.718181,0.643208,0.615199
2,Vinegar Taint,0.294401,0.312273,0.135915
3,SO2,0.243862,0.226176,0.184743
4,pH,0.231216,0.203412,0.429218


## 9 · Save to `generated_explanation/`

Each method gets its own CSV file named `{method}_{dataset}.csv`.  The schema matches
`assets/explanations/coax/attribution.csv` so the files are drop-in compatible with
the `PrecomputedCSVXAIMethod` adapter and the data loader's explanation tables.

In [10]:
to_save = [
    (ks_result     if ks_df     is not None else None, ks_preds,  "rf", f"shap_kernel_{DATASET_ID}.csv"),
    (tree_result   if tree_df   is not None else None, ks_preds,  "rf", f"shap_tree_{DATASET_ID}.csv"),
    (linear_result if linear_df is not None else None, lr_preds,  "lr", f"shap_linear_{DATASET_ID}.csv"),
]

saved = []
for result, preds, model_name, filename in to_save:
    if result is None:
        print(f"  Skipped {filename} (not generated)")
        continue
    path = OUTPUT_DIR / filename
    result.save_csv(path, ids_test, preds, DATASET_ID, model_name)
    saved.append(path)
    print(f"  Saved → tutorials/generated_explanation/{filename}")

print(f"\n{len(saved)} file(s) written to {OUTPUT_DIR}")

NameError: name 'linear_df' is not defined

## 10 · Save Importance CSVs

**Attribution** (signed) records which direction each feature pushed the prediction.
**Importance** (absolute) records how much each feature mattered, regardless of direction —
useful when you only care about magnitude, not sign.

`save_csv(..., importance=True)` passes the absolute values through the same
`i_max` normalization, matching the schema of
`assets/explanations/coax/importance.csv`.

In [ ]:
importance_jobs = [
    (ks_result,     ks_preds, "rf", f"shap_kernel_{DATASET_ID}_importance.csv"),
    (tree_result,   ks_preds, "rf", f"shap_tree_{DATASET_ID}_importance.csv"),
    (linear_result, lr_preds, "lr", f"shap_linear_{DATASET_ID}_importance.csv"),
]

for result, preds, model_name, filename in importance_jobs:
    if result is None:
        print(f"  Skipped {filename} (not generated)")
        continue
    path = OUTPUT_DIR / filename
    result.save_csv(path, ids_test, preds, DATASET_ID, model_name, importance=True)
    print(f"  Saved → tutorials/generated_explanation/{filename}")

# Preview: compare attribution vs importance for KernelSHAP instance 0
if ks_result is not None:
    attr_row = ks_result.to_explanation_df(ids_test, ks_preds, DATASET_ID, "rf").iloc[0]
    imp_row  = ks_result.to_explanation_df(ids_test, ks_preds, DATASET_ID, "rf", importance=True).iloc[0]
    attr_cols = [c for c in attr_row.index if c.startswith("a") and c.endswith("_i")]
    import pandas as pd
    pd.DataFrame({
        "feature":     feature_names,
        "attribution": attr_row[attr_cols].values,
        "importance":  imp_row[attr_cols].values,
    })

## 10 · Reload and Verify

Sanity-check: read back one of the saved files and confirm the schema looks right.

In [9]:
verify_path = OUTPUT_DIR / f"shap_tree_{DATASET_ID}.csv"
if verify_path.exists():
    reloaded = pd.read_csv(verify_path)
    print("Columns:", reloaded.columns.tolist())
    print(f"Shape:   {reloaded.shape}")
    reloaded.head()
else:
    print("File not found — TreeSHAP was skipped above.")

Columns: ['dataId', 'modelName', 'expMethod', 'instanceId', 'pred', 'i_max', 'a0_i', 'a1_i', 'a2_i', 'a3_i', 'a4_i', 'intercept']
Shape:   (24, 12)


---
## Summary

| Step | What happened |
|---|---|
| Data | Loaded `wine_quality` from `assets/ai_dataset/coax/values.csv` |
| Models | Trained RandomForest (RF) and LogisticRegression (LR) |
| KernelSHAP | Model-agnostic; wraps `rf.predict_proba`; approximation |
| TreeSHAP | Exact; `"tree_path_dependent"` mode for RF; no background data needed |
| LinearSHAP | Exact; uses LR coefficients + covariance; scaled features |
| Attribution CSVs | `shap_{kernel,tree,linear}_wine_quality.csv` — signed values |
| Importance CSVs | `shap_{kernel,tree,linear}_wine_quality_importance.csv` — absolute values |

All files land in `tutorials/generated_explanation/` and are compatible with
`PrecomputedCSVXAIMethod` and the data loader's explanation tables.

**API reference**

```python
result.to_explanation_df(instance_ids, predictions, dataset_id, model_name)
result.to_explanation_df(..., importance=True)   # absolute values
result.save_csv(path, instance_ids, predictions, dataset_id, model_name)
result.save_csv(..., importance=True)             # saves importance format
```